# Fine-tune the Kurrent adapter on Kade lines, with SageMaker

This notebook is the **controller**: it runs on a small, cheap CPU notebook instance
(`ml.t3.medium` is enough) and launches the actual training as a SageMaker **training job** on a
GPU instance. The GPU only exists, and only bills, while the job runs. This is the same pattern as
episode 9 of the *ML with AWS SageMaker* lesson, adapted from CHURRO on whole pages to
Qwen3-VL-4B + the Kurrent adapter on single lines. `docs/lora-finetuning-explained.md` explains
every step in plain language.

**Kernel:** `conda_pytorch` (any of the `conda_pytorch_*` kernels).

**Before you start you need:**
1. Your own S3 bucket (lesson episode 2), in the same region as this notebook.
2. `kade_lines.zip`: the line crops from `notebooks/kade_line_segmentation.ipynb` (run on Ubuntu,
   see `docs/ubuntu-setup.md`). Make it there with
   `cd ~/badger-scribe-data && python -m zipfile -c kade_lines.zip kade_lines/`, then drag it into
   this notebook's folder in the Jupyter file browser.
3. A quota for the GPU you pick, for **training job usage** (Service Quotas → SageMaker). The
   default is often 0.

**Run order:** the smoke test first (about 30 minutes, under $1), then the baseline (scores the
Kurrent adapter as it is), then the full run.

## 1. Setup

The same first cell as every lesson episode: a SageMaker session, the region, the role the jobs run as, and an S3 client.

In [ ]:
import json
import os
import sys
import time

import boto3
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

session = sagemaker.Session()
region = session.boto_region_name
role = get_execution_role()
s3_client = boto3.client("s3", region_name=region)
print("Region:", region)
print("Role:", role)
print("SageMaker SDK:", sagemaker.__version__)

Now the names. **Edit the three marked lines.** Your username goes into job names, which have to be unique in the account.

In [ ]:
shared_bucket = "ml-marathon-shared"   # the class's read-only bucket with train.csv and the images
bucket_name = "sinkorswim-doejohn"     # <-- EDIT ME: your own bucket; everything the jobs write goes here
username = "doejohn"                   # <-- EDIT ME: lowercase letters/digits only; used in job names
name, project = "DoeJohn", "BadgerScribe"   # <-- EDIT ME: tags, so the spend can be traced to you

data_prefix = "badger-scribe"
run_prefix = f"{data_prefix}/kade-lora"           # everything this notebook writes, under your bucket

# This notebook lives in <repo>/notebooks; the job's code is in <repo>/sagemaker/code.
repo_dir = os.path.abspath("..")
code_dir = os.path.join(repo_dir, "sagemaker", "code")
sys.path.insert(0, os.path.join(repo_dir, "sagemaker"))
local_data_dir = "/home/ec2-user/SageMaker/kade-lora-data"   # beside the repo, not in it: the Kaggle data stays out of git
os.makedirs(local_data_dir, exist_ok=True)
print(sorted(os.listdir(code_dir)))

## 2. Build the job's input and upload it

This pairs each line crop with its line of the label, for the pages where kraken found the same
number of lines as the label has (the `Postmark` line doesn't count; it isn't in the image).
The dev pages (all of `kade_112`, the team's dev set) are never trained on; they are what the
job scores.

**Check the first printed line:** if well under 70% of training pages are usable, look at
`line_counts.csv` from the segmentation notebook before training.

In [ ]:
import prepare_kade_lines

lines_zip = "kade_lines.zip"   # where you dragged it; change if it is somewhere else
s3_client.download_file(shared_bucket, f"{data_prefix}/train.csv", os.path.join(local_data_dir, "train.csv"))

channel_dir = os.path.join(local_data_dir, "channel")
stats = prepare_kade_lines.build(lines_zip, os.path.join(local_data_dir, "train.csv"), channel_dir)
pd.read_csv(os.path.join(channel_dir, "train_lines.csv"), keep_default_na=False).head()

In [ ]:
# Three files: lines.zip (the crops, unpacked inside the job), train_lines.csv, dev_pages.csv.
train_s3_uri = session.upload_data(path=channel_dir, bucket=bucket_name, key_prefix=f"{run_prefix}/input")
print("train channel:", train_s3_uri)

## 3. Pick the GPU and the run

| Instance | GPU | Price/h* | Notes |
|---|---|---|---|
| `ml.g5.xlarge` | A10G, 24 GB | $1.41 | What the lesson uses. Trains in bf16. |
| `ml.g4dn.xlarge` | T4, 16 GB | $0.74 | Half the price, about 2x slower. The T4 has no bf16, so it trains in fp16, which is more likely to give `nan` losses; if it does, switch to g5. |
| `ml.g6.xlarge` | L4, 24 GB | $1.13 | The lesson's fallback if g5 is short. bf16. |

\*us-east-1 on-demand prices from the lesson; your region may differ slightly.

| Run | What it does | Use it to |
|---|---|---|
| `smoke` | 64 lines, 1 epoch, 2 dev pages | Check everything works before paying for a real run. |
| `baseline` | No training; scores all 11 dev pages | Get the number to beat: the Kurrent adapter as it is. |
| `full` | All lines, 3 epochs, dev scored after every epoch | The real fine-tune. |

Minutes below are estimates until the smoke test gives a real number; `max_run` is a hard cap
(SageMaker stops the job there) at about twice the estimate.

In [ ]:
instance_type = "ml.g5.xlarge"     # or "ml.g4dn.xlarge" / "ml.g6.xlarge"
RUN = "smoke"                      # "smoke", then "baseline", then "full"

prices = {"ml.g5.xlarge": 1.41, "ml.g4dn.xlarge": 0.74, "ml.g6.xlarge": 1.13}
slow = 2.0 if instance_type == "ml.g4dn.xlarge" else 1.0
price = prices[instance_type]

# The knobs. Every key reaches the script as a --flag; the script's parse_args() lists them all.
runs = {
    "smoke":    ({"epochs": 1, "max_train_lines": 64, "dev_pages": 2}, 15),
    "baseline": ({"epochs": 0}, 20),
    "full":     ({"epochs": 3, "lr": 1e-4, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
                  "batch_size": 4, "grad_accum": 4, "min_pixels": 65536}, 45),
}
hyperparameters, base_minutes = runs[RUN]
estimated_minutes = int(base_minutes * slow)
max_run_minutes = 2 * estimated_minutes + 15

run_id = time.strftime("%m%d-%H%M")   # job names must be unique, so every launch gets a new one
training_job_name = f"badger-{username}-kurrent-lora-{RUN}-{run_id}"
checkpoint_s3_uri = f"s3://{bucket_name}/{run_prefix}/checkpoints/{training_job_name}/"

print(f"job name:     {training_job_name}")
print(f"instance:     {instance_type} at ${price}/h")
print(f"estimate:     {estimated_minutes} min  ->  about ${price * estimated_minutes / 60:.2f}")
print(f"max_run cap:  {max_run_minutes} min  ->  worst case ${price * max_run_minutes / 60:.2f}")
print(f"settings:     {hyperparameters}")

## 4. Launch the training job

The estimator says which script to run, in which prebuilt container, on which instance, with
which limits. The container is the lesson's Hugging Face image; its `transformers` 4.56 is too old
for Qwen3-VL (added in 4.57), so `sagemaker/code/requirements.txt` upgrades it when the job starts.

The job downloads Qwen3-VL-4B (~9 GB) and the Kurrent adapter from Hugging Face itself, then
merges them. `wait=True` streams the job's log into this cell. Interrupting the cell does **not**
stop the job; to stop it, run
`session.sagemaker_client.stop_training_job(TrainingJobName=training_job_name)`.

If `from sagemaker.huggingface import HuggingFace` fails, this notebook has a newer SageMaker SDK
than the lesson; `%pip install -q "sagemaker<3"` and restart the kernel.

In [ ]:
from sagemaker.huggingface import HuggingFace

def make_job_tags(purpose):
    return [{"Key": "Name", "Value": name}, {"Key": "Project", "Value": project}, {"Key": "Purpose", "Value": purpose}]

estimator = HuggingFace(
    entry_point="finetune_qwen_kurrent_lora.py",
    source_dir=code_dir,                  # ships the script, kurrent_common.py and requirements.txt
    role=role,
    tags=make_job_tags("Training"),
    instance_type=instance_type,
    instance_count=1,
    volume_size=60,                       # GB of disk: container, 9 GB model, crops
    max_run=max_run_minutes * 60,         # seconds; SageMaker stops the job here no matter what
    checkpoint_s3_uri=checkpoint_s3_uri,  # synced from /opt/ml/checkpoints while the job runs
    checkpoint_local_path="/opt/ml/checkpoints",
    transformers_version="4.56",
    pytorch_version="2.8",
    py_version="py312",
    output_path=f"s3://{bucket_name}/{run_prefix}/models/",
    sagemaker_session=session,
    hyperparameters=hyperparameters,
)
estimator.fit({"train": train_s3_uri}, job_name=training_job_name, wait=True)

## 5. What it cost, and what it scored

Billable time is what you pay for. Compare it with the estimate, and update the estimates in step 3 after the smoke test.

In [ ]:
def job_cost_report(job_name, price_per_hour):
    desc = session.sagemaker_client.describe_training_job(TrainingJobName=job_name)
    billable = desc["BillableTimeInSeconds"]
    print(f"job:            {job_name}")
    print(f"status:         {desc['TrainingJobStatus']}")
    print(f"billable time:  {billable / 60:.1f} min")
    print(f"cost:           ${billable / 3600 * price_per_hour:.3f} at ${price_per_hour}/h")
    tags = session.sagemaker_client.list_tags(ResourceArn=desc["TrainingJobArn"])["Tags"]
    print(f"tags:           {', '.join(t['Key'] + '=' + t['Value'] for t in tags) or 'NONE'}")
    return desc

desc = job_cost_report(training_job_name, price)

In [ ]:
import tarfile

run_dir = os.path.join(local_data_dir, training_job_name)
os.makedirs(run_dir, exist_ok=True)
bucket, key = desc["ModelArtifacts"]["S3ModelArtifacts"].replace("s3://", "").split("/", 1)
s3_client.download_file(bucket, key, os.path.join(run_dir, "model.tar.gz"))
with tarfile.open(os.path.join(run_dir, "model.tar.gz")) as tar:
    tar.extractall(run_dir)

summary = json.load(open(os.path.join(run_dir, "train_summary.json")))
print(json.dumps(summary, indent=2))
print(f"\nKade dev CER: {summary['dev_cer_before']:.3f} before -> {summary['dev_cer_best']:.3f} best "
      f"(epoch {summary['best_epoch']}). Old base-model result: about 0.5.")

### Dev CER after every epoch

Epoch 0 is the Kurrent adapter before any training. If the line goes down and then back up, the
later epochs were memorising the training letters: use fewer epochs or a lower `lr`. The job
already kept the adapter from the best epoch.

In [ ]:
log = json.load(open(os.path.join(run_dir, "train_log.json")))
curve = pd.DataFrame(log["dev_cer_by_epoch"])
print(curve.to_string(index=False))
if log["loss"]:
    import matplotlib.pyplot as plt
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
    loss = pd.DataFrame(log["loss"])
    a.plot(loss.step, loss.loss); a.set_xlabel("step"); a.set_ylabel("training loss")
    b.plot(curve.epoch, curve.dev_cer, marker="o"); b.set_xlabel("epoch"); b.set_ylabel("dev CER (lower is better)")
    plt.tight_layout(); plt.show()

### Page by page

The same dev pages, before training and at the best epoch.

In [ ]:
before = pd.read_csv(os.path.join(run_dir, "eval_epoch0.csv"), keep_default_na=False)
best = pd.read_csv(os.path.join(run_dir, f"eval_epoch{summary['best_epoch']}.csv"), keep_default_na=False)
cmp = before[["page_id", "cer"]].merge(best[["page_id", "cer"]], on="page_id", suffixes=("_before", "_best"))
cmp["gain"] = cmp.cer_before - cmp.cer_best
print(cmp.round(3).to_string(index=False))

page = cmp.sort_values("gain").iloc[-1].page_id   # the page that improved most
print(f"\n{page}\nREFERENCE\n" + before.set_index("page_id").loc[page, "text_ref"][:600])
print("\nBEFORE\n" + before.set_index("page_id").loc[page, "text_pred"][:600])
print("\nBEST\n" + best.set_index("page_id").loc[page, "text_pred"][:600])

## 6. Everything you've spent

Every job from this notebook starts with `badger-<username>-kurrent`, so one listing finds them all.

In [ ]:
total = 0.0
for job in session.sagemaker_client.list_training_jobs(NameContains=f"badger-{username}-kurrent", MaxResults=50)["TrainingJobSummaries"]:
    d = session.sagemaker_client.describe_training_job(TrainingJobName=job["TrainingJobName"])
    cost = d.get("BillableTimeInSeconds", 0) / 3600 * prices.get(d["ResourceConfig"]["InstanceType"], price)
    total += cost
    print(f"{job['TrainingJobName']:<55} {job['TrainingJobStatus']:<11} ${cost:.2f}")
print(f"total: ${total:.2f}")

## 7. When you're done

- Check no job is still `InProgress` (the listing above).
- **Stop this notebook instance** (SageMaker → Notebook instances → Stop). It bills every hour it
  is running, even with the tab closed.
- The adapter is in `/home/ec2-user/SageMaker/kade-lora-data/<job name>/` here and in your bucket under
  `badger-scribe/kade-lora/models/`. It's about 130 MB; the 9 GB base model is never copied.

**Changing settings:** edit the `"full"` entry in step 3 (one setting at a time), then re-run
steps 3 to 5. Each launch gets a new job name and its own checkpoint folder.